# Ensemble: Swin Transformer Tiny + ResNet50Clean, correctly-ordered version. Run cells top to bottom (or Runtime → Run all).

## Cell 1 — Setup (installs + imports)

In [1]:
!pip install tfswin --no-deps -qimport osimport shutilimport zipfileimport numpy as npimport pandas as pdimport tensorflow as tffrom tensorflow.keras import layers, modelsfrom tensorflow.keras.models import load_modelfrom sklearn.metrics import accuracy_score, classification_report, confusion_matriximport matplotlib.pyplot as pltimport seaborn as snsimport tfswin  # must import before loading/rebuilding the Swin modelfrom tfswin import SwinTransformerTiny224print("numpy:", np.__version__)print("pandas:", pd.__version__)print("All imports successful!")

ERROR: Invalid requirement: 'layers,': Expected end or semicolon (after name and no valid version specifier)
    layers,
          ^


## Cell 2 — Drive mount

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

## Cell 3 — Restore test set (images + CSV)

In [ ]:
split_zip_path = '/content/drive/MyDrive/split_dataset.zip'csv_zip_path = '/content/drive/MyDrive/thesis_dataset_csv-20260716T043638Z-1-001.zip'if not os.path.exists('/content/test'):    shutil.copy(split_zip_path, '/content/split_dataset.zip')    with zipfile.ZipFile('/content/split_dataset.zip', 'r') as zip_ref:        zip_ref.extractall('/content/')if not os.path.exists('/content/csv_data'):    with zipfile.ZipFile(csv_zip_path, 'r') as zip_ref:        zip_ref.extractall('/content/csv_data')print("Test folder ready:", os.path.exists('/content/test'))

## Cell 4 — Load test CSV, fix pathsThis defines `class_names`, `label_to_index`, `true_labels` — required by later cells.

In [ ]:
test_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/test_data.csv')test_df['filepath'] = test_df['filepath'].str.replace('/content/split_dataset', '/content')print("Sample file exists:", os.path.exists(test_df['filepath'].iloc[0]))print("Test samples:", len(test_df))class_names = sorted(test_df['label'].unique())num_classes = len(class_names)label_to_index = {name: i for i, name in enumerate(class_names)}true_labels = test_df['label'].map(label_to_index).valuesprint("Classes:", class_names)

## Cell 5 — Config

In [ ]:
IMG_SIZE = (224, 224)BATCH_SIZE = 32filepaths = test_df['filepath'].values

## Cell 6 — Build the two loading functions (different dtype per model)

In [ ]:
# Swin needs raw pixels as uint8def load_uint8(filepath):    img = tf.io.read_file(filepath)    img = tf.image.decode_image(img, channels=3, expand_animations=False)    img = tf.image.resize(img, IMG_SIZE)    img = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)    return img# ResNet50 needs raw pixels as float32 (preprocess_input is baked into the model)def load_float32(filepath):    img = tf.io.read_file(filepath)    img = tf.image.decode_image(img, channels=3, expand_animations=False)    img = tf.image.resize(img, IMG_SIZE)    return img  # stays float32, values 0-255

## Cell 7 — Build the two test datasets

In [ ]:
def make_ds(load_fn):    ds = tf.data.Dataset.from_tensor_slices(filepaths)    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)    return dstest_ds_uint8 = make_ds(load_uint8)      # for Swintest_ds_float32 = make_ds(load_float32)  # for ResNet50print("Datasets ready")

## Cell 8 — Load both trained modelsSwin is rebuilt from architecture + `load_weights` (avoids the `.keras` deserialization bug). ResNet50 is loaded from a zip backup on Drive.

In [ ]:
# --- Swin Transformer Tiny: rebuild architecture, then load trained weights ---inputs = layers.Input(shape=(224, 224, 3), dtype='uint8')base_model = SwinTransformerTiny224(include_top=False)x = base_model(inputs)x = layers.GlobalAveragePooling2D()(x)x = layers.Dense(256, activation='relu')(x)x = layers.Dropout(0.4)(x)outputs = layers.Dense(num_classes, activation='softmax')(x)swin_model = models.Model(inputs, outputs)swin_model.load_weights('/content/drive/MyDrive/thesis_swintiny_outputs/best_swintiny_model.keras')print("Swin model rebuilt and weights loaded!")

In [ ]:
# --- ResNet50: extract from Drive zip backup, then load normally ---resnet_zip_path = '/content/drive/MyDrive/thesis_resnet50_outputs-20260716T043630Z-1-001.zip'print("Zip exists:", os.path.exists(resnet_zip_path))if not os.path.exists('/content/resnet50_data'):    with zipfile.ZipFile(resnet_zip_path, 'r') as zip_ref:        zip_ref.extractall('/content/resnet50_data')resnet_model_path = '/content/resnet50_data/thesis_resnet50_outputs/best_resnet50_model.keras'print("Model file exists:", os.path.exists(resnet_model_path))resnet_model = load_model(resnet_model_path)print("ResNet50 loaded successfully")

## Cell 8b — Sanity check: both models still reproduce their known test accuracy

In [ ]:
pred_swin_check = swin_model.predict(test_ds_uint8, verbose=1)acc_swin_check = accuracy_score(true_labels, np.argmax(pred_swin_check, axis=1))print(f"Swin Tiny test accuracy (rebuilt model): {acc_swin_check:.4f}")pred_resnet_check = resnet_model.predict(test_ds_float32, verbose=1)acc_resnet_check = accuracy_score(true_labels, np.argmax(pred_resnet_check, axis=1))print(f"ResNet50 test accuracy (reloaded model): {acc_resnet_check:.4f}")

## Cell 9 — Get predictions from each model (test set, reused for the ensemble below)

In [ ]:
pred_swin = pred_swin_check      # already computed in the sanity check abovepred_resnet = pred_resnet_check  # already computed in the sanity check above

## Cell 10 — Ensemble (simple average) + accuracy comparison

In [ ]:
ensemble_probs = (pred_swin + pred_resnet) / 2.0ensemble_preds = np.argmax(ensemble_probs, axis=1)pred_swin_labels = np.argmax(pred_swin, axis=1)pred_resnet_labels = np.argmax(pred_resnet, axis=1)acc_swin = accuracy_score(true_labels, pred_swin_labels)acc_resnet = accuracy_score(true_labels, pred_resnet_labels)acc_ensemble = accuracy_score(true_labels, ensemble_preds)print("=" * 55)print("RESULTS (Simple Average Ensemble)")print("=" * 55)print(f"Swin Tiny accuracy:          {acc_swin:.4f}")print(f"ResNet50 accuracy:           {acc_resnet:.4f}")print(f"Ensemble (avg) accuracy:     {acc_ensemble:.4f}")

## Cell 11 — Val set predictions (needed for leakage-free weight tuning)

In [ ]:
val_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/val_data.csv')val_df['filepath'] = val_df['filepath'].str.replace('/content/split_dataset', '/content')val_true_labels = val_df['label'].map(label_to_index).valuesval_filepaths = val_df['filepath'].valuesdef make_ds_from_paths(paths, load_fn):    ds = tf.data.Dataset.from_tensor_slices(paths)    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)    return dsval_ds_uint8 = make_ds_from_paths(val_filepaths, load_uint8)val_ds_float32 = make_ds_from_paths(val_filepaths, load_float32)print("Predicting on VAL set with Swin Tiny...")val_pred_swin = swin_model.predict(val_ds_uint8, verbose=1)print("Predicting on VAL set with ResNet50...")val_pred_resnet = resnet_model.predict(val_ds_float32, verbose=1)

## Cell 12 — Weighted ensemble (weights tuned on VAL set, evaluated once on TEST set — no leakage)

In [ ]:
best_w, best_val_acc = (0.5, 0.5), accuracy_score(    val_true_labels, np.argmax((val_pred_swin + val_pred_resnet) / 2.0, axis=1))for w_swin in np.arange(0.1, 1.0, 0.05):    w_resnet = 1 - w_swin    val_probs = (w_swin * val_pred_swin) + (w_resnet * val_pred_resnet)    val_acc = accuracy_score(val_true_labels, np.argmax(val_probs, axis=1))    if val_acc > best_val_acc:        best_val_acc = val_acc        best_w = (round(w_swin, 2), round(w_resnet, 2))print(f"Best weighted combo (found on VAL set): {best_w}")print(f"Val accuracy with this combo: {best_val_acc:.4f}")# Apply this fixed weight to the TEST set — only ONE evaluation, no more searchingw_swin_final, w_resnet_final = best_wtest_probs_weighted = (w_swin_final * pred_swin) + (w_resnet_final * pred_resnet)ensemble_preds_weighted = np.argmax(test_probs_weighted, axis=1)acc_ensemble_weighted = accuracy_score(true_labels, ensemble_preds_weighted)print(f"\nFinal TEST accuracy with tuned weights: {acc_ensemble_weighted:.4f}")

## Cell 13 — Classification report (using the simple-average ensemble predictions)

In [ ]:
report = classification_report(true_labels, ensemble_preds, target_names=class_names)print(report)

## Cell 14 — Confusion matrix + save everything to Drive

In [ ]:
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs'os.makedirs(output_folder, exist_ok=True)with open(f'{output_folder}/ensemble_swin_resnet50_report.txt', 'w') as f:    f.write(f"Swin Tiny accuracy: {acc_swin:.4f}\n")    f.write(f"ResNet50 accuracy: {acc_resnet:.4f}\n")    f.write(f"Ensemble (simple average) accuracy: {acc_ensemble:.4f}\n")    f.write(f"Best weighted combo (tuned on val set): {best_w}\n")    f.write(f"Ensemble (weighted) test accuracy: {acc_ensemble_weighted:.4f}\n\n")    f.write(report)cm = confusion_matrix(true_labels, ensemble_preds)plt.figure(figsize=(12, 10))sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Purples')plt.xlabel('Predicted')plt.ylabel('True')plt.title('Ensemble (Swin Tiny + ResNet50) Confusion Matrix')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.savefig(f'{output_folder}/ensemble_confusion_matrix.png', dpi=150, bbox_inches='tight')plt.show()print(f"Results saved to: {output_folder}")